In [0]:
%run ../config

In [0]:
%run ../helpers

# Import

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import IntegerType, DoubleType

# Ingestion

In [0]:
df_bdr = spark.read.table("chicago_crime.bronze.community_area_boundaries")

# Data Profiling

## Explore Sample

In [0]:
display(
    df_bdr.limit(10)
)

## Explore Schema

In [0]:
df_bdr.printSchema()

## Explore Overview

In [0]:
display(
    df_bdr.agg(
        F.count("*").alias("total_row"),
        F.countDistinct("area_numbe").alias("distinct_codes"),
        F.min(F.expr("try_cast(area_numbe as int)")).alias("min_code"),
        F.max(F.expr("try_cast(area_numbe as int)")).alias("max_code"),
        F.min(F.expr("try_cast(shape_area as double)")).alias("min_shape_area"),
        F.max(F.expr("try_cast(shape_area as double)")).alias("max_shape_area"),
        F.min(F.expr("try_cast(shape_len as double)")).alias("min_shape_len"),
        F.max(F.expr("try_cast(shape_len as double)")).alias("max_shape_len")
    )
)

# Transform

In [0]:
df_bdr_silver = (df_bdr
    .withColumn("community_area_code", col("area_numbe").try_cast(IntegerType()))
    .withColumn("community_area_name", col("community"))
    .withColumn("boundary_geometry", col("the_geom"))
    .withColumn("shape_area_sqft", col("shape_area").try_cast(DoubleType()))
    .withColumn("shape_perimeter_ft", col("shape_len").try_cast(DoubleType()))
    .withColumn("_processed_at", current_timestamp())
    .select(
        "community_area_code",
        "community_area_name",
        "boundary_geometry",
        "shape_area_sqft",
        "shape_perimeter_ft",
        "_processed_at",
    )
)

# Data Quality

In [0]:
EXPECTED_ROWS = 77

stats = df_bdr_silver.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("community_area_code").alias("total_distinct_codes"),
    F.sum(F.col("community_area_code").isNull().cast("int")).alias("total_null_codes"),
    F.sum((~F.col("community_area_code").between(1, 77)).cast("int")).alias("total_out_of_range_codes"),
    F.sum(F.col("community_area_name").isNull().cast("int")).alias("total_null_names"),
    F.sum(F.col("shape_area_sqft").isNull().cast("int")).alias("total_null_area"),
    F.sum(F.col("shape_perimeter_ft").isNull().cast("int")).alias("total_null_perimeter"),
    F.sum(F.get_json_object("boundary_geometry", "$.type").isNull().cast("int")).alias("bad_geometry")
).collect()[0]

In [0]:
errors = []

if stats["total_rows"] != EXPECTED_ROWS:
    errors.append(f"Expected {EXPECTED_ROWS} rows, found {stats['total_rows']}")
if stats["total_distinct_codes"] != stats["total_rows"] - stats["total_null_codes"]:
    errors.append("duplicated community_area_code")

for check_name in ["total_null_codes", "total_out_of_range_codes", "total_null_names", "total_null_area", "total_null_perimeter", "bad_geometry"]:
    if stats[check_name] > 0:
        errors.append(f"{check_name} = {stats[check_name]}")

if errors:
    raise ValueError("Silver boundaries DQ failed: \n" + "\n".join(errors))

print("[DQ] passed:", stats.asDict())

# Write to Silver

In [0]:
TARGET_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.community_area_boundaries"

(
    df_bdr_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TARGET_TABLE)
)

print(f"Loaded {spark.table(TARGET_TABLE).count()} rows into {TARGET_TABLE}")

# Verify

## Explore Schema

In [0]:
spark.table(TARGET_TABLE).printSchema()

## Explore Overview

In [0]:
display(
    spark.table(TARGET_TABLE).agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("community_area_code").alias("distinct_codes")
    )
)

## Explore Sample

In [0]:
display(
    spark.table(TARGET_TABLE).limit(10)
)